In [ ]:
import torch
import numpy as np
import random

def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

seed_everything(42)

In [ ]:
import sys
from pathlib import Path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


from IPython import get_ipython
from pprint import pformat
formatter = get_ipython().display_formatter.formatters['text/plain']
formatter.for_type(dict, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))
formatter.for_type(list, lambda obj, p, _: p.text(pformat(obj, width=120, sort_dicts=False)))

<function IPython.lib.pretty._seq_pprinter_factory.<locals>.inner(obj, p, cycle)>

## Evaluation Process (Sequential + Milestones)

This notebook evaluates sequential knowledge editing with milestone checkpoints.

1. Build paired data as `(EditRequest, EvalSuite)` for each dataset.
2. Shuffle and select `n = eval_milestones[-1]` pairs.
3. Split selected pairs into aligned lists:
   - `edit_requests[i]`
   - `eval_suites[i]`
4. Apply edits sequentially from step `1` to `max(eval_milestones)`.
5. At each milestone step `k`:
   - Evaluate on all stored suites up to that step: `eval_suites[:k]`.
   - Aggregate returned per-suite `EvalResult` objects into one milestone result.
6. Save outputs to:
   - `data/evaluation/{method}/{model}_{dataset}_{milestone1-milestone2-...}.json`

### Metric usage by dataset

- `cfact`: `success` + `locality`
- `mquake`: `portability`

### Why evaluate all previous suites at each milestone?

This checks both:
- immediate edit quality on newer edits, and
- retention/stability of older edits after more sequential updates.

In [ ]:
from statistics import mean

from src.editors import EditRequest
from src.evaluate import EvalRecord, MetricResult, EvalSuite, EvalResult


def select_and_split_pairs(pairs: list[tuple[EditRequest, EvalSuite]],
                           n: int) -> tuple[list[EditRequest], list[EvalSuite]]:
    if n <= 0:
        raise ValueError("n must be > 0")
    if len(pairs) < n:
        raise ValueError(f"Requested n={n}, but only {len(pairs)} pairs are available.")

    shuffled = pairs.copy()
    random.shuffle(shuffled)
    selected = shuffled[:n]

    edit_requests = [request for request, _ in selected]
    eval_suites = [suite for _, suite in selected]
    return edit_requests, eval_suites


def aggregate_eval_results(results: list[EvalResult]) -> EvalResult:
    if not results:
        return EvalResult()

    def _aggregate_metric(metric_name: str) -> MetricResult | None:
        metric_results = [getattr(result, metric_name) for result in results]
        metric_results = [metric for metric in metric_results if metric is not None]
        if not metric_results:
            return None

        details: list[EvalRecord] = [
            record
            for metric in metric_results
            for record in metric.details
        ]
        score = mean(metric.score for metric in metric_results)

        return MetricResult(score=score, details=details)

    return EvalResult(
        success=_aggregate_metric("success"),
        locality=_aggregate_metric("locality"),
        portability=_aggregate_metric("portability"),
    )


In [ ]:
def build_cfact_pairs(records) -> list[tuple[EditRequest, EvalSuite]]:
    pairs: list[tuple[EditRequest, EvalSuite]] = []
    return pairs


def build_mquake_pairs(records) -> list[tuple[EditRequest, EvalSuite]]:
    pairs: list[tuple[EditRequest, EvalSuite]] = []
    return pairs

In [ ]:
from dataclasses import asdict
from typing import Literal

from tqdm.notebook import trange

from src.evaluate import evaluate


async def run_sequential_milestone_benchmark(
    editor,
    edit_requests: list[EditRequest],
    eval_suites: list[EvalSuite],
    eval_milestones: list[int],
    eval_method: Literal["teacher_forcing", "llm_judge"],
    eval_batch_size: int,
    **kwargs
) -> list[dict]:

    if not eval_milestones:
        raise ValueError("eval_milestones must not be empty")

    eval_milestones = sorted(eval_milestones)
    if eval_milestones[0] <= 0:
        raise ValueError("Milestones must be positive integers")

    max_step = eval_milestones[-1]
    if len(edit_requests) < max_step or len(eval_suites) < max_step:
        raise ValueError(
            f"Need at least {max_step} requests/suites, got {len(edit_requests)} requests and {len(eval_suites)} suites."
        )

    milestones = set(eval_milestones)
    milestone_results: list[dict] = []

    for step in trange(1, max_step + 1, desc="sequential benchmark"):
        editor.edit(edit_requests[step - 1])

        if step not in milestones:
            continue

        step_results = await evaluate(
            editor.model,
            editor.tokenizer,
            eval_suites[:step],
            method=eval_method,
            batch_size=eval_batch_size,
            **kwargs
        )

        milestone_results.append({
            "step": step,
            "result": asdict(aggregate_eval_results(step_results))
        })

    return milestone_results

In [ ]:
# --- New benchmark configuration ---
editing_method = ["ft", "rome", "grace", "wise", "alpha_edit"][4]
model_name = ["qwen2.5-7b"][0]

eval_method = "teacher_forcing"  # or "llm_judge"
eval_batch_size = 16

eval_milestones = [1, 5, 10]
n_records = eval_milestones[-1]

model_generation_config = {
    "max_new_tokens": 128,
    # "do_sample": True,
    # "temperature": 0.7,
    # "top_p": 0.8,
    # "top_k": 20,
}
llm_judge_config = {
    "max_concurrency": 5,
    "max_retries": 3,
    # "init_retry_delay": 1.0,
    # "max_retry_delay": 10.0,
}

In [ ]:
from datasets import load_from_disk

from src.editors import load_editor


# --- Load editor + datasets ---
editor = load_editor(f"../src/config/{editing_method}/{model_name}.yaml")
cfact = load_from_disk("../data/datasets/cfact")
mquake = load_from_disk("../data/datasets/mquake")

cfact_pairs = build_cfact_pairs(cfact)
mquake_pairs = build_mquake_pairs(mquake)

cfact_requests, cfact_suites = select_and_split_pairs(cfact_pairs, n_records)
mquake_requests, mquake_suites = select_and_split_pairs(mquake_pairs, n_records)

len(cfact_requests), len(cfact_suites), len(mquake_requests), len(mquake_suites)

In [ ]:
import json
from pathlib import Path


EVALUATION_DIR = "../data/evaluation"
def save_benchmark_results(
    results: list[dict],
    method: str,
    model: str,
    dataset: str,
    milestones: list[int],
    eval_dir: str | None = None
) -> Path:

    if not milestones:
        raise ValueError("milestones must not be empty")

    milestones_str = "-".join(str(milestone) for milestone in milestones)
    dest = Path(eval_dir if eval_dir else EVALUATION_DIR)
    dest = dest / method
    dest.mkdir(parents=True, exist_ok=True)

    path = dest / f"{model}_{dataset}_{milestones_str}.json"
    with open(path, "w", encoding="utf-8") as file:
        json.dump(results, file, indent=2, ensure_ascii=False)

    return path

True

In [ ]:
# --- Run cfact benchmark (success + locality) ---
editor.reset()

cfact_milestone_results = await run_sequential_milestone_benchmark(
    editor=editor,
    edit_requests=cfact_requests,
    eval_suites=cfact_suites,
    eval_milestones=eval_milestones,
    eval_method=eval_method,
    eval_batch_size=eval_batch_size,
    model_generation_config=model_generation_config,
    llm_judge_config=llm_judge_config,
)

cfact_save_path = save_benchmark_results(
    results=cfact_milestone_results,
    method=editing_method,
    model=model_name,
    dataset="cfact",
    milestones=eval_milestones,
)

cfact_save_path
cfact_milestone_results

In [ ]:
# --- Run mquake benchmark (portability) ---
editor.reset()

mquake_milestone_results = await run_sequential_milestone_benchmark(
    editor=editor,
    edit_requests=mquake_requests,
    eval_suites=mquake_suites,
    eval_milestones=eval_milestones,
    eval_method=eval_method,
    eval_batch_size=eval_batch_size,
    model_generation_config=model_generation_config,
    llm_judge_config=llm_judge_config,
)

mquake_save_path = save_benchmark_results(
    results=mquake_milestone_results,
    method=editing_method,
    model=model_name,
    dataset="mquake",
    milestones=eval_milestones,
)

mquake_save_path
mquake_milestone_results

## Visualization

Use these cells after running benchmarks and saving result files.

In [ ]:
from src.visualize import table_results, plot_results

In [ ]:
table_results(eval_milestones)

In [ ]:
plot_results(eval_milestones)